In [1]:
import os
import random
import numpy as np
import torch
import cv2
import matplotlib.pyplot as plt
from ultralytics import YOLO

Para que el codigo sea replicable por cualqueir persona se crea la función set_seed para garantizar la reproducibilidad de los resultados obtenidos

In [2]:
def set_seed(seed = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [3]:
modelo = os.getenv("YOLO_MODEL", "yolov8n.pt")
confianza = float(os.getenv("YOLO_CONF_THRESHOLD", "0.25"))
class_id = int(os.getenv("YOLO_DOG_CLASS_ID", "16"))


In [18]:
modelo_yolo = YOLO(modelo)

In [19]:
def procesar_imagen_pipeline(ruta_imagen, modelo_y, umbral_confianza=0.25):
    # 1. Usuario carga una imagen (la leemos a memoria)
    imagen = cv2.imread(ruta_imagen)
    
    if imagen is None:
        print("Error al cargar la imagen.")
        return None, []

    # 2. YOLO detecta todos los objetos en la imagen cargada
    # Le pasamos la matriz de la imagen directamente en lugar de una ruta
    resultados = modelo_y.predict(source = imagen, conf = umbral_confianza, save = False)
    
    recortes_para_clasificar = []
    
    # 3. Se generan bounding boxes (iteramos los resultados)
    for r in resultados:
        for box in r.boxes:
            clase_id = int(box.cls[0].item())
            confianza = box.conf[0].item()
            
            # Filtramos para asegurarnos de que es un perro (Clase 16)
            if clase_id == 16:
                # Extraemos las coordenadas y las convertimos a enteros básicos
                x1, y1, x2, y2 = [int(c) for c in box.xyxy[0].tolist()]
                
                # 4. Se recortan las regiones detectadas
                # Explicación: En OpenCV, las imágenes son matrices. 
                # Para recortar, indicamos [fila_inicio : fila_fin, columna_inicio : columna_fin]
                # En términos de coordenadas esto es [y1:y2, x1:x2]
                recorte = imagen[y1:y2, x1:x2]
                
                # Verificamos que el recorte no esté vacío (por si las coordenadas tocan un borde)
                if recorte.size > 0:
                    # Guardamos el recorte en memoria para el paso 5
                    recortes_para_clasificar.append({
                        "imagen_recortada": recorte,
                        "coordenadas": (x1, y1, x2, y2),
                        "score_yolo": confianza
                    })
                
                # Preparamos la visualización para el usuario final
                # Dibujamos la bounding box y el score en la imagen ORIGINAL
                texto_etiqueta = f"Perro detectado: {confianza:.2f}"
                cv2.rectangle(imagen, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(imagen, texto_etiqueta, (x1, y1 - 10), 
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

    # Retornamos la imagen entera (ya con las cajas dibujadas) y la lista de recortes
    return imagen, recortes_para_clasificar

In [20]:
imagen_procesada, lista_perros = procesar_imagen_pipeline("data\dataset\test\Afghan\04.jpg", modelo_yolo, confianza)

Error al cargar la imagen.


<>:1: SyntaxWarning: invalid escape sequence '\d'
<>:1: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Nicolas\AppData\Local\Temp\ipykernel_14732\962555445.py:1: SyntaxWarning: invalid escape sequence '\d'
  imagen_procesada, lista_perros = procesar_imagen_pipeline("data\dataset\test\Afghan\04.jpg", modelo_yolo, confianza)
